In [1]:
!wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py

--2024-06-24 08:26:15--  https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3832 (3,7K) [text/plain]
Saving to: 'minsearch.py.1'

     0K ...                                                   100% 6,80M=0,001s

2024-06-24 08:26:15 (6,80 MB/s) - 'minsearch.py.1' saved [3832/3832]



In [2]:
import minsearch

In [3]:
import json

In [4]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [5]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [6]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [7]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

SELECT * WHERE course = 'data-engineering-zoomcamp';

In [8]:
q = 'the course has already started, can I still enroll?'

In [9]:
index.fit(documents)

In [10]:
def search(query):
    boost = {'question': 3, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5
    )

    return results

In [11]:
search(q)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 202

In [12]:
import os
from groq import Groq
from dotenv import load_dotenv

# Завантаження змінних середовища з файлу .env
load_dotenv()

# Убедитесь, что переменная окружения GROQ_API_KEY установлена
api_key = os.environ.get("GROQ_API_KEY")

if api_key is None:
    raise ValueError("API key is not set. Please set the GROQ_API_KEY environment variable.")

client = Groq(api_key=api_key)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama3-8b-8192",
)

print(chat_completion.choices[0].message.content)


Fast language models have gained significant importance in recent years due to their ability to process and generate human-like language at scale, speed, and efficiency. Here are some key reasons why fast language models are crucial:

1. **Real-time processing**: Fast language models can process language inputs in real-time, enabling applications like chatbots, virtual assistants, and language translators to respond quickly to user queries. This responsiveness is crucial for building engaging user experiences.
2. **Efficient computation**: Fast language models are optimized for parallel processing and can handle large volumes of data, reducing the computational resources required for tasks like text classification, sentiment analysis, and language translation.
3. **Scalability**: As data volumes continue to grow, fast language models can handle increased data, making them essential for applications like natural language processing (NLP) and machine learning (ML) pipelines.
4. **Improve

In [13]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [14]:
def llm(prompt):
    response = client.chat.completions.create(
        model='llama3-8b-8192',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [15]:
query = 'how do I run kafka?'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [16]:
print(rag(query))

A QUESTION!

Based on the CONTEXT, I'll try to provide a most helpful answer...

 QUESTION: how do I run kafka?

From the CONTEXT, I see that Kafka is mentioned in several sections, including Module 6: Streaming with Kafka. However, I don't see a direct answer on how to run Kafka.

But I can try to infer... The Python question "Module “kafka” not found when trying to run producer.py" mentions Kafka, and the answer suggests that you need to create a virtual environment and install required packages. The same section has an answer "Run this command in terminal in the same directory (/docker/spark): chmod +x build.sh" which might be related to building or running Kafka.

If I had to guess, I'd say that the way to run Kafka depends on whether you're using Java or Python.

If you're using Java, you can try running `java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java` as mentioned in another question.

If you're using Python, you might need to set 

In [17]:
rag('the course has already started, can I still enroll?')

'Based on the context provided, the question is: the course has already started, can I still enroll?\n\nFrom the FAQ database, I found the following relevant information:\n\n* The section "Course - Can I still join the course after the start date?" states: "Yes, even if you don\'t register, you\'re still eligible to submit the homeworks..."'

In [18]:
print(documents[0])

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.", 'section': 'General course-related questions', 'question': 'Course - When will the course start?', 'course': 'data-engineering-zoomcamp'}


In [19]:
from elasticsearch import Elasticsearch

In [20]:
es_client = Elasticsearch('http://localhost:9200') 

In [21]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

index_name = "course-questions"

es_client.indices.create(index=index_name,
                         settings=index_settings['settings'],
                         mappings=index_settings['mappings'])

BadRequestError: BadRequestError(400, 'resource_already_exists_exception', 'index [course-questions/auyjNtXASf-28i5g83stJQ] already exists')

In [22]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [23]:
from tqdm.auto import tqdm

In [24]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

ConnectionTimeout: Connection timed out

In [25]:
query = 'I just disovered the course. Can I still join it?'

In [26]:
query = 'how do I run kafka?'
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name,
                                size=search_query['size'],
                                # sort=search_query['sort'],
                                query=search_query['query'])
    
    result_docs = []
    
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])
    
    return result_docs

In [27]:
elastic_search(query)
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

ApiError: ApiError(503, 'search_phase_execution_exception', None)

In [28]:
print(rag(query))

Based on the provided CONTEXT, to run Kafka, you have the following options:

* For Java Kafka, run the producer/consumer/kstreams/etc in the terminal by running the command `java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java` in the project directory.
* For Python Kafka, ensure that you're in the correct virtual environment, and then run the command without any additional information. If you're facing difficulties running the Python code, refer to the other sections for relevant solutions, such as creating a virtual environment or changing permissions on the `build.sh` file.

Note that there is no single "how to run Kafka" instruction that applies universally to all cases. Each scenario has its unique solution, which is reflected in the provided answers.
